## Grover basado en MindQuantum

In [1]:
# pylint: disable=W0104
from mindquantum import Circuit, UN, H, Z
from mindquantum.simulator import Simulator, SUPPORTED_SIMULATOR


In [2]:
SUPPORTED_SIMULATOR.base_module

{'mqvector': <module 'mindquantum._mq_vector' from '/home/alfredov/miniconda3/envs/mindquantum/lib/python3.11/site-packages/mindquantum/_mq_vector.cpython-311-x86_64-linux-gnu.so'>,
 'mqmatrix': <module 'mindquantum._mq_matrix' from '/home/alfredov/miniconda3/envs/mindquantum/lib/python3.11/site-packages/mindquantum/_mq_matrix.cpython-311-x86_64-linux-gnu.so'>,
 'stabilizer': <module 'mindquantum._mq_stabilizer' from '/home/alfredov/miniconda3/envs/mindquantum/lib/python3.11/site-packages/mindquantum/_mq_stabilizer.cpython-311-x86_64-linux-gnu.so'>}

In [3]:

n_qubits = 3                                 # Set the number of qubits to 3
sim = Simulator('mqvector_gpu', n_qubits)        # Use the projectq simulator, named sim

circuit = Circuit()                          # Initialize the quantum circuit, named circuit
circuit += UN(H, n_qubits)                   # H-gate operations are performed on each qubit

sim.apply_circuit(circuit)                   # Run the built quantum circuit circuit by the simulator sim

circuit                                      # Print the quantum circuit circuit at this time


ValueError: GPU backend is not available due to: Malloc GPU memory failed: cudaErrorInsufficientDriver, CUDA driver version is insufficient for CUDA runtime version

In [ ]:
def bitphaseflip_operator(phase_inversion_qubit, n_qubits):   # Define a function that flips the phase of a qubit
    s = [1 for i in range(1 << n_qubits)]
    for i in phase_inversion_qubit:
        s[i] = -1
    if s[0] == -1:
        for i in range(len(s)):
            s[i] = -1 * s[i]
    circuit = Circuit()
    length = len(s)
    cz = []
    for i in range(length):
        if s[i] == -1:
            cz.append([])
            current = i
            t = 0
            while current != 0:
                if (current & 1) == 1:
                    cz[-1].append(t)
                t += 1
                current = current >> 1
            for j in range(i + 1, length):
                if i & j == i:
                    s[j] = -1 * s[j]
    for i in cz:
        if i:
            if len(i) > 1:
                circuit += Z.on(i[-1], i[:-1])
            else:
                circuit += Z.on(i[0])

    return circuit


In [ ]:
print(sim.get_qs(True))                      # Print the final state after running the quantum circuit circuit in the simulator sim


√2/4¦000⟩
√2/4¦001⟩
√2/4¦010⟩
√2/4¦011⟩
√2/4¦100⟩
√2/4¦101⟩
√2/4¦110⟩
√2/4¦111⟩


In [ ]:
# pylint: disable=W0104
sim.reset()                                                      # Reset the quantum state maintained by the simulator sim so that the initialized quantum state is |000>

phase_inversion_qubit = [4]                                      # Invert the phase of the |4> state
operator = bitphaseflip_operator(phase_inversion_qubit, n_qubits)# Call our defined bitphaseflip_operator() function

circuit += operator                                              # Add the quantum gate required to flip the phase of the |4> state in the quantum circuit circuit

sim.apply_circuit(circuit)                                       # Run the built quantum circuit circuit again by the simulator sim

circuit                                                          # Print the quantum circuit circuit at this time


┏━━━┓                           
q0: ──┨ H ┠─────────■───────────■─────
      ┗━━━┛         ┃           ┃     
      ┏━━━┓         ┃           ┃     
q1: ──┨ H ┠─────────╂─────■─────■─────
      ┗━━━┛         ┃     ┃     ┃     
      ┏━━━┓ ┏━━━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓   
q2: ──┨ H ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠───
      ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛

In [ ]:
print(sim.get_qs(True))                                          # Print the final state after running the quantum circuit circuit in the simulator sim


√2/4¦000⟩
√2/4¦001⟩
√2/4¦010⟩
√2/4¦011⟩
-√2/4¦100⟩
√2/4¦101⟩
√2/4¦110⟩
√2/4¦111⟩


In [ ]:
print(int('100', 2))


4


In [ ]:
# pylint: disable=W0104
n_qubits = 3                                                     # Set the number of qubits to 3
sim1 = Simulator('mqvector_gpu', n_qubits)                           # Use the projectq simulator, named sim1

operator1 = bitphaseflip_operator([i for i in range(1, pow(2, n_qubits))], n_qubits) # Call our defined bitphaseflip_operator() function to flip the phase of each state except |0> state, named operator1

circuit1 = Circuit()                                             # Initialize the quantum circuit, named circuit1
circuit1 += UN(H, n_qubits)                                      # H-gate operations are performed on each qubit
circuit1 += operator1                                            # Add the quantum gates required to flip the phase of each state except the |0> state in the quantum circuit circuit1

sim1.apply_circuit(circuit1)                                     # Run the built quantum circuit circuit1 by the simulator sim1

circuit1                                                         # Print the quantum circuit circuit1 at this time


┏━━━┓ ┏━━━┓                           
q0: ──┨ H ┠─┨ Z ┠───■─────■───────────■─────
      ┗━━━┛ ┗━━━┛   ┃     ┃           ┃     
      ┏━━━┓ ┏━━━┓ ┏━┻━┓   ┃           ┃     
q1: ──┨ H ┠─┨ Z ┠─┨ Z ┠───╂─────■─────■─────
      ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     ┃     ┃     
      ┏━━━┓ ┏━━━┓       ┏━┻━┓ ┏━┻━┓ ┏━┻━┓   
q2: ──┨ H ┠─┨ Z ┠───────┨ Z ┠─┨ Z ┠─┨ Z ┠───
      ┗━━━┛ ┗━━━┛       ┗━━━┛ ┗━━━┛ ┗━━━┛

In [ ]:
print(sim1.get_qs(True))                                         # Print the final state after running the quantum circuit circuit1 in the simulator sim1


√2/4¦000⟩
-√2/4¦001⟩
-√2/4¦010⟩
-√2/4¦011⟩
-√2/4¦100⟩
-√2/4¦101⟩
-√2/4¦110⟩
-√2/4¦111⟩


In [ ]:
def G(phase_inversion_qubit, n_qubits):           # Define the G operator in Grover search algorithm
    operator = bitphaseflip_operator(phase_inversion_qubit, n_qubits)
    operator += UN(H, n_qubits)
    operator += bitphaseflip_operator([i for i in range(1, pow(2, n_qubits))], n_qubits)
    operator += UN(H, n_qubits)
    return operator

In [ ]:
# pylint: disable=W0104
from numpy import pi, sqrt

n_qubits = 3                                      # Set the number of qubits to 3
phase_inversion_qubit = [2]                       # Set the target state that needs to flip the phase, and flip the phase of the |2> state here

N = 2 ** (n_qubits)                               # Calculate the total number of elements in the database
M = len(phase_inversion_qubit)                    # Calculate the total number of target states

r = int(pi / 4 * sqrt(N / M))                     # Set the number of iterations of the G operator to r

sim2 = Simulator('mqvector_gpu', n_qubits)            # Use the projectq simulator, named sim2

circuit2 = Circuit()                              # Initialize the quantum circuit, named circuit2
circuit2 += UN(H, n_qubits)                       # H-gate operations are performed on each qubit

for i in range(r):                                # Execute the G operator r times in a loop
    circuit2 += G(phase_inversion_qubit, n_qubits)

sim2.apply_circuit(circuit2)                      # Run the built quantum circuit circuit2 by the simulator sim2

circuit2                                          # Print the quantum circuit circuit2 at this time


┏━━━┓                         ┏━━━┓ ┏━━━┓                         ┏━━━┓         
q0: ──┨ H ┠─────────■───────────■───┨ H ┠─┨ Z ┠───■─────■───────────■───┨ H ┠───────[red bold]↯[/]─
      ┗━━━┛         ┃           ┃   ┗━━━┛ ┗━━━┛   ┃     ┃           ┃   ┗━━━┛         
      ┏━━━┓ ┏━━━┓ ┏━┻━┓         ┃   ┏━━━┓ ┏━━━┓ ┏━┻━┓   ┃           ┃   ┏━━━┓ ┏━━━┓   
q1: ──┨ H ┠─┨ Z ┠─┨ Z ┠───■─────■───┨ H ┠─┨ Z ┠─┨ Z ┠───╂─────■─────■───┨ H ┠─┨ Z ┠─[red bold]↯[/]─
      ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     ┃   ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     ┃     ┃   ┗━━━┛ ┗━━━┛   
      ┏━━━┓             ┏━┻━┓ ┏━┻━┓ ┏━━━┓ ┏━━━┓       ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━━━┓         
q2: ──┨ H ┠─────────────┨ Z ┠─┨ Z ┠─┨ H ┠─┨ Z ┠───────┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ H ┠───────[red bold]↯[/]─
      ┗━━━┛             ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛       ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛         
                                                                                      
                        ┏━━━┓ ┏━━━┓                         ┏━━━┓                     
q0: ────■───────────■───┨ H ┠─┨ Z ┠───■─────■───────────■───┨ H ┠───                  
        ┃           ┃   ┗━━━┛ ┗━━━┛   ┃     ┃           ┃   ┗━━━┛                     
      ┏━┻━┓         ┃   ┏━━━┓ ┏━━━┓ ┏━┻━┓   ┃           ┃   ┏━━━┓                     
q1: ──┨ Z ┠───■─────■───┨ H ┠─┨ Z ┠─┨ Z ┠───╂─────■─────■───┨ H ┠───                  
      ┗━━━┛   ┃     ┃   ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     ┃     ┃   ┗━━━┛                     
            ┏━┻━┓ ┏━┻━┓ ┏━━━┓ ┏━━━┓       ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━━━┓                     
q2: ────────┨ Z ┠─┨ Z ┠─┨ H ┠─┨ Z ┠───────┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ H ┠───                  
            ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛       ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛

In [ ]:
print(sim2.get_qs(True))                          # Print the final state after running the quantum circuit circuit2 in the simulator sim2


-√2/16¦000⟩
-√2/16¦001⟩
0.97227182¦010⟩
-√2/16¦011⟩
-√2/16¦100⟩
-√2/16¦101⟩
-√2/16¦110⟩
-√2/16¦111⟩


In [ ]:
# pylint: disable=W0104
from mindquantum import Measure

sim2.reset()                                      # Reset the quantum state maintained by the simulator sim2, so that the initialized quantum state is |000>

circuit2 += UN(Measure(), circuit2.n_qubits)      # Add a measurement gate to each qubit in the quantum circuit circuit2

result = sim2.sampling(circuit2, shots=1000)      # Sampling the quantum circuit circuit2 1000 times by the simulator sim2
result                                            # print sampling results


shots: 1000
Keys: q2 q1 q0│0.00     0.2         0.4         0.6         0.8         1.0
──────────────┼───────────┴───────────┴───────────┴───────────┴───────────┴
           000│▒
              │
           001│▒
              │
           010│▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
              │
           011│▒
              │
           100│▒
              │
           101│▒
              │
           110│▒
              │
           111│▒
              │
{'000': 14, '001': 8, '010': 934, '011': 7, '100': 11, '101': 8, '110': 9, '111': 9}

In [ ]:
print(int('010', 2))
  

2


## Ejemplo 2


In [ ]:
# pylint: disable=W0104
n_qubits = 5                                      # Set the number of qubits to 5
phase_inversion_qubit = [5, 11]                   # Set the target state to be phase-flipped, and flip the phases of the |5> state and |11> state here

N = 2 ** (n_qubits)                               # Calculate the total number of elements in the database
M = len(phase_inversion_qubit)                    # Calculate the total number of target states

r = int(pi / 4 * sqrt(N / M))                     # Set the number of iterations of the G operator to r

sim3 = Simulator('mqvector_gpu', n_qubits)            # Use the projectq simulator, named sim3

circuit3 = Circuit()                              # Initialize the quantum circuit, named circuit3
circuit3 += UN(H, n_qubits)                       # H-gate operations are performed on each qubit

for i in range(r):                                # Execute the G operator r times in a loop
    circuit3 += G(phase_inversion_qubit, n_qubits)

sim3.apply_circuit(circuit3)                      # Run the built quantum circuit circuit3 by the simulator sim3

circuit3                                          # Print the quantum circuit circuit3 at this time


┏━━━┓                                                 ┏━━━┓ ┏━━━┓               
q0: ──┨ H ┠───■─────■─────■─────■─────■─────■─────■─────■───┨ H ┠─┨ Z ┠───■─────■───[red bold]↯[/]─
      ┗━━━┛   ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃   ┗━━━┛ ┗━━━┛   ┃     ┃     
      ┏━━━┓   ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃   ┏━━━┓ ┏━━━┓ ┏━┻━┓   ┃     
q1: ──┨ H ┠───╂─────■─────■─────╂─────╂─────■─────■─────╂───┨ H ┠─┨ Z ┠─┨ Z ┠───╂───[red bold]↯[/]─
      ┗━━━┛   ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃   ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     
      ┏━━━┓ ┏━┻━┓ ┏━┻━┓   ┃     ┃     ┃     ┃     ┃     ┃   ┏━━━┓ ┏━━━┓       ┏━┻━┓   
q2: ──┨ H ┠─┨ Z ┠─┨ Z ┠───╂─────■─────■─────■─────╂─────■───┨ H ┠─┨ Z ┠───────┨ Z ┠─[red bold]↯[/]─
      ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     ┃     ┃     ┃     ┃     ┃   ┗━━━┛ ┗━━━┛       ┗━━━┛   
      ┏━━━┓             ┏━┻━┓ ┏━┻━┓   ┃     ┃     ┃     ┃   ┏━━━┓                     
q3: ──┨ H ┠─────────────┨ Z ┠─┨ Z ┠───╂─────╂─────■─────■───┨ H ┠───────────────────[red bold]↯[/]─
      ┗━━━┛             ┗━━━┛ ┗━━━┛   ┃     ┃     ┃     ┃   ┗━━━┛                     
      ┏━━━┓                         ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━━━┓                     
q4: ──┨ H ┠─────────────────────────┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ H ┠───────────────────[red bold]↯[/]─
      ┗━━━┛                         ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛                     
                                                                                      
q0: ──────────■─────■───────────■───────────■───────────■─────■───────────■─────────[red bold]↯[/]─
              ┃     ┃           ┃           ┃           ┃     ┃           ┃           
              ┃     ┃           ┃           ┃           ┃     ┃           ┃           
q1: ────■─────■─────╂─────■─────■───────────╂─────■─────■─────╂─────■─────■─────────[red bold]↯[/]─
        ┃     ┃     ┃     ┃     ┃           ┃     ┃     ┃     ┃     ┃     ┃           
      ┏━┻━┓ ┏━┻━┓   ┃     ┃     ┃           ┃     ┃     ┃     ┃     ┃     ┃           
q2: ──┨ Z ┠─┨ Z ┠───╂─────╂─────╂─────■─────■─────■─────■─────╂─────╂─────╂─────■───[red bold]↯[/]─
      ┗━━━┛ ┗━━━┛   ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     
      ┏━━━┓       ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓   ┃     ┃     ┃     ┃     
q3: ──┨ Z ┠───────┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠───╂─────╂─────╂─────╂───[red bold]↯[/]─
      ┗━━━┛       ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛   ┃     ┃     ┃     ┃     
      ┏━━━┓                                                 ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓   
q4: ──┨ Z ┠─────────────────────────────────────────────────┨ Z ┠─┨ Z ┠─┨ Z ┠─┨ Z ┠─[red bold]↯[/]─
      ┗━━━┛                                                 ┗━━━┛ ┗━━━┛ ┗━━━┛ ┗━━━┛   
                                                                                      
                                                                        ┏━━━┓         
q0: ────■───────────■───────────■───────────■───────────■───────────■───┨ H ┠───■───[red bold]↯[/]─
        ┃           ┃           ┃           ┃           ┃           ┃   ┗━━━┛   ┃     
        ┃           ┃           ┃           ┃           ┃           ┃   ┏━━━┓   ┃     
q1: ────╂─────■─────■───────────╂─────■─────■───────────╂─────■─────■───┨ H ┠───╂───[red bold]↯[/]─
        ┃     ┃     ┃           ┃     ┃     ┃           ┃     ┃     ┃   ┗━━━┛   ┃     
        ┃     ┃     ┃           ┃     ┃     ┃           ┃     ┃     ┃   ┏━━━┓ ┏━┻━┓   
q2: ────■─────■─────■───────────╂─────╂─────╂─────■─────■─────■─────■───┨ H ┠─┨ Z ┠─[red bold]↯[/]─
        ┃     ┃     ┃           ┃     ┃     ┃     ┃     ┃     ┃     ┃   ┗━━━┛ ┗━━━┛   
        ┃     ┃     ┃           ┃     ┃     ┃     ┃     ┃     ┃     ┃   ┏━━━┓         
q3: ────╂─────╂─────╂─────■─────■─────■─────■─────■─────■─────■─────■───┨ H ┠───────[red bold]↯[/]─
        ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃     ┃   ┗━━━┛         
      ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━┻━┓ ┏━━━┓      

In [ ]:
print(sim3.get_qs(True))                          # Print the final state after running the quantum circuit circuit3 in the simulator sim3


-0.03590777¦00000⟩
-0.03590777¦00001⟩
-0.03590777¦00010⟩
-0.03590777¦00011⟩
-0.03590777¦00100⟩
0.6932961¦00101⟩
-0.03590777¦00110⟩
-0.03590777¦00111⟩
-0.03590777¦01000⟩
-0.03590777¦01001⟩
-0.03590777¦01010⟩
0.6932961¦01011⟩
-0.03590777¦01100⟩
-0.03590777¦01101⟩
-0.03590777¦01110⟩
-0.03590777¦01111⟩
-0.03590777¦10000⟩
-0.03590777¦10001⟩
-0.03590777¦10010⟩
-0.03590777¦10011⟩
-0.03590777¦10100⟩
-0.03590777¦10101⟩
-0.03590777¦10110⟩
-0.03590777¦10111⟩
-0.03590777¦11000⟩
-0.03590777¦11001⟩
-0.03590777¦11010⟩
-0.03590777¦11011⟩
-0.03590777¦11100⟩
-0.03590777¦11101⟩
-0.03590777¦11110⟩
-0.03590777¦11111⟩


In [ ]:

# pylint: disable=W0104
sim3.reset()                                      # Reset the quantum state maintained by the simulator sim3 so that the initialized quantum state is |00000>

circuit3 += UN(Measure(), circuit3.n_qubits)      # Add a measurement gate to each qubit in the quantum circuit circuit3

result1 = sim3.sampling(circuit3, shots=1000)     # The quantum circuit circuit3 is sampled 1000 times by the simulator sim3
result1                                           # print sampling results


shots: 1000
Keys: q4 q3 q2 q1 q0│0.00   0.124       0.248       0.372       0.496        0.62
────────────────────┼───────────┴───────────┴───────────┴───────────┴───────────┴
               00000│▒
                    │
               00001│▒
                    │
               00011│▒
                    │
               00101│▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
                    │
               00110│▒
                    │
               00111│▒
                    │
               01001│▒
                    │
               01011│▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
                    │
               01101│▒
                    │
               10000│▒
                    │
               10001│▒
                    │
               10010│▒
                    │
               10011│▒
                    │
               10100│▒
                    │
               10101│▒
                    │
               10111│▒
                    │
               11000│▒
                    │
               11001│▒
                    │
               11010│▒
                    │
               11011│▒
                    │
               11110│▒
                    │
{'00000': 1, '00001': 4, '00011': 1, '00101': 474, '00110': 3, '00111': 1, '01001': 1, '01011': 496, '01101': 1, '10000': 1, '10001': 2, '10010': 2, '10011': 1, '10100': 2, '10101': 1, '10111': 3, '11000': 2, '11001': 1, '11010': 1, '11011': 1, '11110': 1}